# Adult Income — Feature Engineering and EDA
## Project 3

**Objective:** Understand the Adult Income data, identify hidden missing values, perform EDA,
engineer useful features, and export analysis-ready CSV files.

> The included CSV is a **5,000-row practice dataset with the official 15-column Adult schema**.
> Replace it with the official Kaggle `adult.csv` when required.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
BASE=Path.cwd()
if not (BASE/"dataset").exists(): BASE=BASE.parent
DATA=BASE/"dataset"/"adult_income.csv"; OUT=BASE/"outputs"; OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA); display(df.head()); print("Shape:",df.shape)

## 1. Data Information and Missing Markers

In [ ]:
print(df.info())
categorical=["workclass","education","marital.status","occupation","relationship","race","sex","native.country"]
marker_counts={c:int((df[c]=="?").sum()) for c in categorical}
display(pd.DataFrame.from_dict(marker_counts,orient="index",columns=["? count"]))

## 2. Convert `?` to Missing Values

In [ ]:
for c in categorical: df[c]=df[c].replace("?",np.nan)
missing=pd.DataFrame({"column":df.columns,"missing_count":df.isna().sum().values,"missing_percentage":(df.isna().mean()*100).round(2).values})
display(missing); missing.to_csv(OUT/"missing_values_after_conversion.csv",index=False)

## 3. Income Distribution

In [ ]:
target=df["income"].value_counts()
display(target.to_frame("count"))
plt.figure(figsize=(8,5)); plt.bar(target.index,target.values); plt.title("Income Class Distribution"); plt.xlabel("Income"); plt.ylabel("Count"); plt.show()

## 4. Age Distribution

In [ ]:
plt.figure(figsize=(8,5)); plt.hist(df["age"],bins=20); plt.title("Age Distribution"); plt.xlabel("Age"); plt.ylabel("Count"); plt.show()

## 5. Education vs Income

In [ ]:
rate=df.groupby("education.num")["income"].apply(lambda s:(s==">50K").mean())
plt.figure(figsize=(9,5)); plt.plot(rate.index,rate.values,marker="o"); plt.title("Education Number vs >50K Income Rate"); plt.xlabel("Education Number"); plt.ylabel("Proportion >50K"); plt.show()
rate.to_csv(OUT/"education_number_income_rate.csv",header=["income_gt_50k_rate"])

## 6. Occupation vs Income

In [ ]:
rate=df.groupby("occupation")["income"].apply(lambda s:(s==">50K").mean()).sort_values()
plt.figure(figsize=(10,6)); plt.barh(rate.index,rate.values); plt.title("Income >50K Rate by Occupation"); plt.xlabel("Proportion >50K"); plt.show()

## 7. Feature Engineering

In [ ]:
df["has_capital_gain"]=(df["capital.gain"]>0).astype(int)
df["has_capital_loss"]=(df["capital.loss"]>0).astype(int)
df["net_capital"]=df["capital.gain"]-df["capital.loss"]
df["age_hours"]=df["age"]*df["hours.per.week"]
df["capital_gain_log1p"]=np.log1p(df["capital.gain"])
df["capital_loss_log1p"]=np.log1p(df["capital.loss"])
df["age_group"]=pd.cut(df["age"],[17,25,35,45,55,65,100],labels=["18-25","26-35","36-45","46-55","56-65","66+"])
display(df[["age","hours.per.week","age_hours","capital.gain","capital_gain_log1p","net_capital","has_capital_gain","has_capital_loss","age_group"]].head(10))

## 8. Capital Gain Transformation

In [ ]:
plt.figure(figsize=(8,5)); plt.hist(df["capital_gain_log1p"],bins=30); plt.title("log1p(Capital Gain)"); plt.xlabel("log1p(Capital Gain)"); plt.ylabel("Count"); plt.show()

## 9. Hours per Week vs Income

In [ ]:
hr=df.groupby("hours.per.week")["income"].apply(lambda s:(s==">50K").mean())
plt.figure(figsize=(9,5)); plt.plot(hr.index,hr.values); plt.title("Hours per Week vs >50K Income Rate"); plt.xlabel("Hours per Week"); plt.ylabel("Proportion >50K"); plt.show()

## 10. Numerical Summary and Correlation

In [ ]:
nums=["age","fnlwgt","education.num","capital.gain","capital.loss","hours.per.week","has_capital_gain","has_capital_loss","net_capital","age_hours","capital_gain_log1p","capital_loss_log1p"]
display(df[nums].describe().T)
display(df[nums].corr())

## 11. Export Final CSV Outputs

In [ ]:
cleaned=df.copy()
for c in categorical: cleaned[c]=cleaned[c].fillna("Unknown")
cleaned.to_csv(OUT/"adult_income_cleaned.csv",index=False)
df.to_csv(OUT/"adult_income_feature_engineered.csv",index=False)
df[nums].describe().T.to_csv(OUT/"numeric_summary.csv")
df[nums].corr().to_csv(OUT/"numeric_correlation_matrix.csv")
print("All main CSV outputs saved to:",OUT)

## Conclusion

The project completes the EDA and feature-engineering stage. Important transformations include
handling `?` as missing information, creating capital-presence indicators, net capital,
the age-hours interaction, age groups, and `log1p` versions of skewed capital variables.

These outputs are ready for a later classification project.